# 面试问题：API Rate Limiting 怎样组合 Token Bucket 与 Sliding Window，既允许小突发又阻止窗口边界穿透？

        ## 可直接复述的回答主线

        1. 固定窗口按请求数计费简单，但窗口边界两侧可以瞬间放过双倍流量，而且无法区分批量推理与单条预测成本。
2. Token Bucket 按租户保存容量、余额和上次补充时间，允许受控突发并按请求 cost 扣 Token。
3. Sliding Window 保存最近一段时间的已放行成本，阻止多个租户在边界附近把共享下游压垮。
4. 组合策略先预览 refill 后余额和滚动成本，两个门禁都通过才原子扣减与入窗。
5. 结果应逐请求展示 allow、原因、余额、滚动成本，并比较同一流量下的过载窗口数量。
6. 生产还要处理分布式原子性、单调时钟、Redis/Lua、分片热键、Retry-After、租户配额和降级策略。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是 free/pro 两个租户在模型 API 上的十四次请求，包含单条预测、批量 embedding 和 rerank。每条记录有毫秒时间、业务 endpoint 和计算成本；成本为教学单位，来自脱敏离线事件，不代表真实 GPU 定价。

In [1]:
from collections import deque  # 使用标准库双端队列保存滚动窗口事件。
requests = [{"id": "limit-01", "time_ms": 900, "tenant": "free", "endpoint": "/embed/batch", "cost": 4}, {"id": "limit-02", "time_ms": 940, "tenant": "free", "endpoint": "/predict", "cost": 1}, {"id": "limit-03", "time_ms": 970, "tenant": "pro", "endpoint": "/rerank", "cost": 3}, {"id": "limit-04", "time_ms": 990, "tenant": "pro", "endpoint": "/predict", "cost": 1}, {"id": "limit-05", "time_ms": 1005, "tenant": "free", "endpoint": "/embed/batch", "cost": 4}, {"id": "limit-06", "time_ms": 1020, "tenant": "pro", "endpoint": "/embed/batch", "cost": 4}, {"id": "limit-07", "time_ms": 1100, "tenant": "free", "endpoint": "/predict", "cost": 1}, {"id": "limit-08", "time_ms": 1200, "tenant": "pro", "endpoint": "/predict", "cost": 1}, {"id": "limit-09", "time_ms": 1450, "tenant": "free", "endpoint": "/rerank", "cost": 3}, {"id": "limit-10", "time_ms": 1600, "tenant": "pro", "endpoint": "/predict", "cost": 1}, {"id": "limit-11", "time_ms": 1905, "tenant": "free", "endpoint": "/predict", "cost": 1}, {"id": "limit-12", "time_ms": 1960, "tenant": "pro", "endpoint": "/rerank", "cost": 3}, {"id": "limit-13", "time_ms": 2010, "tenant": "free", "endpoint": "/predict", "cost": 1}, {"id": "limit-14", "time_ms": 2100, "tenant": "pro", "endpoint": "/predict", "cost": 1}]  # 定义十四条跨窗口边界且成本不同的 API 请求。
tenant_limits = {"free": {"capacity": 6.0, "refill_per_second": 2.0}, "pro": {"capacity": 8.0, "refill_per_second": 3.0}}  # 定义两个租户的 Token Bucket 配额。
global_window_ms = 1000  # 设定共享下游的一秒滚动窗口。
global_cost_limit = 10  # 设定滚动窗口最多放行十个计算成本单位。
print("教学实验输入：模型 API 限流事件")  # 标记下方为确定性离线流量。
print("请求       time_ms  tenant  endpoint          cost")  # 输出请求预览表头。
for request in requests:  # 逐条展示到达时间、租户和成本。
    print(f"{request['id']:<10} {request['time_ms']:>7}  {request['tenant']:<7} {request['endpoint']:<17} {request['cost']:>4}")  # 输出当前请求业务字段。
print("租户Bucket=", tenant_limits, "global sliding limit=", global_cost_limit)  # 展示策略输入而不是隐藏常量。

教学实验输入：模型 API 限流事件
请求       time_ms  tenant  endpoint          cost
limit-01       900  free    /embed/batch         4
limit-02       940  free    /predict             1
limit-03       970  pro     /rerank              3
limit-04       990  pro     /predict             1
limit-05      1005  free    /embed/batch         4
limit-06      1020  pro     /embed/batch         4
limit-07      1100  free    /predict             1
limit-08      1200  pro     /predict             1
limit-09      1450  free    /rerank              3
limit-10      1600  pro     /predict             1
limit-11      1905  free    /predict             1
limit-12      1960  pro     /rerank              3
limit-13      2010  free    /predict             1
limit-14      2100  pro     /predict             1
租户Bucket= {'free': {'capacity': 6.0, 'refill_per_second': 2.0}, 'pro': {'capacity': 8.0, 'refill_per_second': 3.0}} global sliding limit= 10


## 2. Baseline / 基线：固定一秒窗口，每租户最多四个请求

基线按 `time_ms // 1000` 分桶，只数请求、不看 cost。900–990ms 与 1005–1200ms 分属不同窗口，因此边界附近的高成本请求会全部放行。

In [2]:
def fixed_window_limit(events, max_requests=4, window_ms=1000):  # 手写按租户和自然时间桶计数的基线限流器。
    counters = {}  # 保存每个租户、窗口的请求计数。
    rows = []  # 保存逐请求决策。
    for request in events:  # 按到达顺序处理同一批事件。
        window_id = request["time_ms"] // window_ms  # 用整数除法确定固定窗口编号。
        key = (request["tenant"], window_id)  # 构造租户隔离的计数键。
        count_before = counters.get(key, 0)  # 读取当前窗口已放行请求数。
        allowed = count_before < max_requests  # 只按请求数量判断是否放行。
        if allowed:  # 对放行请求更新固定窗口计数。
            counters[key] = count_before + 1  # 原子语义下递增当前计数。
        rows.append({"id": request["id"], "time_ms": request["time_ms"], "tenant": request["tenant"], "cost": request["cost"], "allowed": allowed, "reason": "count_ok" if allowed else "fixed_window_count", "window_id": window_id, "count_before": count_before})  # 保存当前决策和中间计数。
    return rows  # 返回全部基线决策。
def rolling_cost_violations(events, decisions, window_ms, cost_limit):  # 检查任意真实滚动窗口内的已放行成本是否超限。
    accepted = deque()  # 保存仍位于窗口内的放行事件。
    violations = []  # 保存每个超限时刻和滚动成本。
    event_by_id = {event["id"]: event for event in events}  # 建立请求索引供决策回读。
    for decision in decisions:  # 按时间检查每条决策。
        event = event_by_id[decision["id"]]  # 读取当前请求成本和时间。
        while accepted and accepted[0][0] <= event["time_ms"] - window_ms:  # 移除窗口左边界之外的旧事件。
            accepted.popleft()  # 弹出过期成本记录。
        if decision["allowed"]:  # 只把真实放行请求计入共享负载。
            accepted.append((event["time_ms"], event["cost"], event["id"]))  # 记录当前放行成本。
        rolling_cost = sum(item[1] for item in accepted)  # 计算最近一秒总计算成本。
        if rolling_cost > cost_limit:  # 检查共享下游是否被穿透。
            violations.append({"at": event["id"], "time_ms": event["time_ms"], "rolling_cost": rolling_cost, "members": [item[2] for item in accepted]})  # 保存可解释超限窗口。
    return violations  # 返回全部过载证据。
baseline_rows = fixed_window_limit(requests)  # 对十四条流量执行固定窗口基线。
baseline_violations = rolling_cost_violations(requests, baseline_rows, global_window_ms, global_cost_limit)  # 评估真实滚动成本安全性。
print("Baseline 固定窗口决策")  # 标记下表展示边界穿透。
print("请求       window  count_before  cost  allowed  reason")  # 输出基线决策表头。
for row in baseline_rows:  # 逐请求展示固定计数状态。
    print(f"{row['id']:<10} {row['window_id']:>6} {row['count_before']:>13} {row['cost']:>5} {str(row['allowed']):>8}  {row['reason']}")  # 输出当前请求的窗口和决策。
print("Baseline滚动成本超限：", baseline_violations)  # 展示固定窗口实际造成的过载区间。

Baseline 固定窗口决策
请求       window  count_before  cost  allowed  reason
limit-01        0             0     4     True  count_ok
limit-02        0             1     1     True  count_ok
limit-03        0             0     3     True  count_ok
limit-04        0             1     1     True  count_ok
limit-05        1             0     4     True  count_ok
limit-06        1             0     4     True  count_ok
limit-07        1             1     1     True  count_ok
limit-08        1             1     1     True  count_ok
limit-09        1             2     3     True  count_ok
limit-10        1             2     1     True  count_ok
limit-11        1             3     1     True  count_ok
limit-12        1             3     3     True  count_ok
limit-13        2             0     1     True  count_ok
limit-14        2             0     1     True  count_ok
Baseline滚动成本超限： [{'at': 'limit-05', 'time_ms': 1005, 'rolling_cost': 13, 'members': ['limit-01', 'limit-02', 'limit-03', 'limit-04', 

## 3. 底层实现：租户 Token Bucket 加全局 Sliding Window

每次请求先按经过时间补充租户余额，再清理全局一秒窗口。只有余额和滚动成本都足够时才同时扣 Token、写入窗口；拒绝不会消耗额度。

In [3]:
def hybrid_limit(events, tenant_config, window_ms, cost_limit):  # 手写 Token Bucket 与 Sliding Window 的组合限流器。
    bucket_state = {tenant: {"tokens": config["capacity"], "last_ms": events[0]["time_ms"]} for tenant, config in tenant_config.items()}  # 初始化每个租户满桶和补充时钟。
    accepted_window = deque()  # 保存全局滚动窗口内已放行成本。
    rows = []  # 保存逐请求余额、滚动成本和决策。
    for request in events:  # 按时间顺序处理每个请求。
        state = bucket_state[request["tenant"]]  # 读取当前租户桶状态。
        config = tenant_config[request["tenant"]]  # 读取容量和补充速率。
        elapsed_ms = max(request["time_ms"] - state["last_ms"], 0)  # 用非负单调时间差计算补充量。
        refill = elapsed_ms / 1000.0 * config["refill_per_second"]  # 把经过毫秒转换为新增 Token。
        state["tokens"] = min(config["capacity"], state["tokens"] + refill)  # 补充但不超过桶容量。
        state["last_ms"] = request["time_ms"]  # 更新当前租户最近访问时刻。
        while accepted_window and accepted_window[0][0] <= request["time_ms"] - window_ms:  # 清理滚动窗口之外的成本。
            accepted_window.popleft()  # 移除过期放行事件。
        rolling_before = sum(item[1] for item in accepted_window)  # 计算请求到达前共享滚动成本。
        bucket_ok = state["tokens"] + 1.0e-12 >= request["cost"]  # 检查租户是否有足够成本 Token。
        window_ok = rolling_before + request["cost"] <= cost_limit  # 检查放行后共享下游是否超限。
        allowed = bucket_ok and window_ok  # 要求租户与全局两个门禁同时通过。
        reason = "allowed" if allowed else "bucket_empty" if not bucket_ok else "sliding_window_full"  # 给出唯一首要拒绝原因。
        tokens_before_charge = state["tokens"]  # 保存扣减前余额供结果解释。
        if allowed:  # 仅对实际放行请求提交两个状态变更。
            state["tokens"] -= request["cost"]  # 从当前租户桶扣除请求成本。
            accepted_window.append((request["time_ms"], request["cost"], request["id"]))  # 把成本加入共享滚动窗口。
        rolling_after = sum(item[1] for item in accepted_window)  # 计算决策后的真实共享成本。
        rows.append({"id": request["id"], "time_ms": request["time_ms"], "tenant": request["tenant"], "cost": request["cost"], "allowed": allowed, "reason": reason, "refill": refill, "tokens_before": tokens_before_charge, "tokens_after": state["tokens"], "rolling_before": rolling_before, "rolling_after": rolling_after})  # 保存完整中间状态。
    return rows  # 返回组合限流决策。
corrected_rows = hybrid_limit(requests, tenant_limits, global_window_ms, global_cost_limit)  # 对同一十四条流量运行组合策略。
print("组合限流前八条中间状态")  # 标记下表展示 refill、余额和滚动成本。
print("请求       refill  token_before/after  rolling_before/after  allowed  reason")  # 输出核心中间量表头。
for row in corrected_rows[:8]:  # 展示边界附近最关键八条请求。
    print(f"{row['id']:<10} {row['refill']:>6.2f} {row['tokens_before']:>7.2f}/{row['tokens_after']:<7.2f} {row['rolling_before']:>7}/{row['rolling_after']:<7} {str(row['allowed']):>8}  {row['reason']}")  # 输出当前请求的状态迁移。

组合限流前八条中间状态
请求       refill  token_before/after  rolling_before/after  allowed  reason
limit-01     0.00    6.00/2.00          0/4           True  allowed
limit-02     0.08    2.08/1.08          4/5           True  allowed
limit-03     0.21    8.00/5.00          5/8           True  allowed
limit-04     0.06    5.06/4.06          8/9           True  allowed
limit-05     0.13    1.21/1.21          9/9          False  bucket_empty
limit-06     0.09    4.15/4.15          9/9          False  sliding_window_full
limit-07     0.19    1.40/0.40          9/10          True  allowed
limit-08     0.54    4.69/4.69         10/10         False  sliding_window_full


## 4. 逐请求结果与结果解读

同一批请求上比较放行与原因，并再次用独立滚动检查器计算过载。组合策略会拒绝部分高成本边界请求，但继续允许后续低成本请求。

In [4]:
corrected_violations = rolling_cost_violations(requests, corrected_rows, global_window_ms, global_cost_limit)  # 独立验证组合策略是否仍有过载窗口。
baseline_allowed_cost = sum(row["cost"] for row in baseline_rows if row["allowed"])  # 汇总固定窗口放行成本。
corrected_allowed_cost = sum(row["cost"] for row in corrected_rows if row["allowed"])  # 汇总安全组合策略放行成本。
corrected_cheap_allowed = sum(row["allowed"] and row["cost"] == 1 for row in corrected_rows)  # 统计低成本请求的可用性。
print("请求       tenant cost  fixed_window            hybrid")  # 输出逐请求同数据对照表头。
for baseline, corrected in zip(baseline_rows, corrected_rows):  # 逐条比较固定窗口和组合策略。
    print(f"{baseline['id']:<10} {baseline['tenant']:<7} {baseline['cost']:>4}  {str(baseline['allowed']):>5}/{baseline['reason']:<20} {str(corrected['allowed']):>5}/{corrected['reason']}")  # 输出当前请求的两种决策。
print(f"结果解读：固定窗口放行成本={baseline_allowed_cost}且产生{len(baseline_violations)}个滚动超限时刻；组合策略放行成本={corrected_allowed_cost}、超限={len(corrected_violations)}，仍放行{corrected_cheap_allowed}条低成本请求。")  # 解释安全与可用性的权衡。

请求       tenant cost  fixed_window            hybrid
limit-01   free       4   True/count_ok              True/allowed
limit-02   free       1   True/count_ok              True/allowed
limit-03   pro        3   True/count_ok              True/allowed
limit-04   pro        1   True/count_ok              True/allowed
limit-05   free       4   True/count_ok             False/bucket_empty
limit-06   pro        4   True/count_ok             False/sliding_window_full
limit-07   free       1   True/count_ok              True/allowed
limit-08   pro        1   True/count_ok             False/sliding_window_full
limit-09   free       3   True/count_ok             False/bucket_empty
limit-10   pro        1   True/count_ok             False/sliding_window_full
limit-11   free       1   True/count_ok              True/allowed
limit-12   pro        3   True/count_ok              True/allowed
limit-13   free       1   True/count_ok              True/allowed
limit-14   pro        1   True/count_ok    

## 5. 失败案例与修正：窗口边界双倍突发

取真实事件中 900–1020ms 的六条请求。固定窗口在 1000ms 换桶后重置计数，高成本请求继续放行；Sliding Window 仍保留前一秒成本并拒绝穿透。

In [5]:
boundary_requests = requests[:6]  # 选择跨越 1000ms 边界的六条真实 API 事件。
boundary_fixed = fixed_window_limit(boundary_requests, max_requests=4)  # 用自然窗口计数处理边界突发。
boundary_hybrid = hybrid_limit(boundary_requests, tenant_limits, global_window_ms, global_cost_limit)  # 用滚动成本处理同一突发。
boundary_fixed_cost = sum(row["cost"] for row in boundary_fixed if row["allowed"])  # 计算固定窗口实际放行成本。
boundary_hybrid_cost = sum(row["cost"] for row in boundary_hybrid if row["allowed"])  # 计算组合策略安全放行成本。
boundary_fixed_violations = rolling_cost_violations(boundary_requests, boundary_fixed, global_window_ms, global_cost_limit)  # 检查固定窗口边界过载。
boundary_hybrid_violations = rolling_cost_violations(boundary_requests, boundary_hybrid, global_window_ms, global_cost_limit)  # 检查组合策略边界过载。
print(f"错误行为：固定窗口跨边界放行成本={boundary_fixed_cost}，滚动超限={boundary_fixed_violations}")  # 展示自然窗口重置造成的穿透。
print(f"修正行为：Token Bucket+Sliding Window放行成本={boundary_hybrid_cost}，滚动超限={boundary_hybrid_violations}")  # 展示组合门禁消除过载。

错误行为：固定窗口跨边界放行成本=17，滚动超限=[{'at': 'limit-05', 'time_ms': 1005, 'rolling_cost': 13, 'members': ['limit-01', 'limit-02', 'limit-03', 'limit-04', 'limit-05']}, {'at': 'limit-06', 'time_ms': 1020, 'rolling_cost': 17, 'members': ['limit-01', 'limit-02', 'limit-03', 'limit-04', 'limit-05', 'limit-06']}]
修正行为：Token Bucket+Sliding Window放行成本=9，滚动超限=[]


## 6. 生产边界

单进程字典没有分布式竞争。生产中要用 Redis Lua 或数据库条件更新保证 refill、扣减和滑窗写入原子；还需单调时钟、跨区域配额、热 key 分片、故障时 fail-open/closed 选择、Retry-After、动态成本模型和租户申诉审计。

In [6]:
rate_limit_diagnostics = {"requests": len(requests), "tenants": len(tenant_limits), "baseline_allowed_cost": baseline_allowed_cost, "hybrid_allowed_cost": corrected_allowed_cost, "baseline_rolling_violations": len(baseline_violations), "hybrid_rolling_violations": len(corrected_violations), "cheap_requests_allowed": corrected_cheap_allowed, "distributed_atomicity": False}  # 汇总容量、安全和教学实现边界。
print("生产监控快照：", rate_limit_diagnostics)  # 输出限流服务应持续跟踪的指标。

生产监控快照： {'requests': 14, 'tenants': 2, 'baseline_allowed_cost': 29, 'hybrid_allowed_cost': 16, 'baseline_rolling_violations': 10, 'hybrid_rolling_violations': 0, 'cheap_requests_allowed': 6, 'distributed_atomicity': False}


## 7. 最小回归测试

断言覆盖样本规模、余额边界、滚动安全、低成本可用性和真实边界失败。

In [7]:
assert len(requests) >= 6 and len(tenant_limits) == 2  # 保证案例包含足够业务请求和多个租户。
assert all(0.0 <= row["tokens_after"] <= tenant_limits[row["tenant"]]["capacity"] for row in corrected_rows)  # 保证 Token Bucket 余额始终在合法范围。
assert len(baseline_violations) > 0 and len(corrected_violations) == 0  # 保证固定窗口过载真实发生且组合策略消除。
assert corrected_cheap_allowed >= 5  # 保证安全门禁没有把全部低成本请求一起拒绝。
assert boundary_fixed_cost > global_cost_limit and boundary_fixed_violations  # 保证跨自然窗口的双倍突发可复现。
assert boundary_hybrid_cost <= global_cost_limit and not boundary_hybrid_violations  # 保证 Sliding Window 修正边界穿透。